In [1]:
import json
import argparse
import tqdm
import time
import textstat

import re

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import prettytable
from prettytable import PrettyTable
from scipy.stats import spearmanr, pearsonr, kendalltau


In [2]:
summeval_fp = '/users/data/summeval.json'
prompt_fp1 = '/users/prompts/summeval/coh_detailed.txt'

prompt_fp1 = '/users/prompts/summeval/rel_detailed.txt'
save_fp = '/users/results/gpt4_con_detailed_openai.json'

input_fp_rel = '/users/results/gpt4_rel_detailed.json'
input_fp_coh = '/users/results/gpt4_coh_detailed.json'




In [3]:
def parse_output(output):
    matched = re.search("^ ?([\d\.]+)", output)
    if (matched):
        try:
            score = float(matched.group(1))
        except:
            score = 0
    else:
        score = 0
    return score

<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_1282739/2072338213.py:2: SyntaxWarning: invalid escape sequence '\d'
  matched = re.search("^ ?([\d\.]+)", output)


In [5]:
def measure_all_scores(input_fp, dimension='relevance'):
    ''' Measure linguistic complexity and store results sequentially with index '''
    import json
    import textstat

    with open(input_fp, 'r', encoding='utf-8') as f:
        jobj = json.load(f)

    model_scores = []
    human_scores = []
    readability_scores = []

    for idx, item in enumerate(jobj):
        # Model score
        all_responses = item["all_responses"]
        all_scores = [parse_output(x) for x in all_responses]
        score = sum(all_scores) / len(all_scores)
        model_scores.append(score)

        # Human score
        human_scores.append(item['scores'][dimension])

        # Readability measurement
        paragraph = item.get("system_output", "").strip()
        if paragraph:
            readability = {
                'flesch_reading_ease': textstat.flesch_reading_ease(paragraph),
                'flesch_kincaid_grade': textstat.flesch_kincaid_grade(paragraph),
                'gunning_fog': textstat.gunning_fog(paragraph),
                'smog_index': textstat.smog_index(paragraph),
                'automated_readability_index': textstat.automated_readability_index(paragraph),
                'dale_chall_score': textstat.dale_chall_readability_score(paragraph),
                'difficult_words': textstat.difficult_words(paragraph),
                'lexicon_count': textstat.lexicon_count(paragraph),
                'syllable_count': textstat.syllable_count(paragraph),
            }
        else:
            readability = {}  # or optionally: None or a dict of NaNs

        readability_scores.append(readability)

    return model_scores, human_scores, readability_scores


In [6]:
# point: all red_scores should be the same; they are computed using different files, which are supposed to have identical source and output_staments

model_scores_rel,human_scores_rel, red_scores_rel = measure_all_scores(input_fp_rel, dimension = 'relevance')
model_scores_coh,human_scores_coh, red_scores_coh = measure_all_scores(input_fp_coh, dimension = 'coherence')
